In [ ]:
# ============================================================
# Notebook:
# 01_competitive_offers_pdf_to_bronze_silver_review.ipynb
#
# Project:
# SDI Competitive Offers PDF Extraction
#
# Purpose:
# This notebook processes one weekly Competitive Offer Report PDF.
#
# It creates:
# 1. Bronze tables:
#    - raw PDF metadata
#    - raw slide/page text
#    - raw slide/page lines
#    - detected entities like money, dates, devices, carriers
#
# 2. Silver tables:
#    - slide section classification
#    - offer candidates
#    - normalized offer records
#    - offer-device bridge for grouped device offers
#
# 3. Review tables:
#    - rows needing human review
#    - auto-approved rows
#    - taxonomy table for categories
#    - approved examples table for future learning
#
# Important:
# - This notebook does NOT create one table per slide.
# - One bronze table stores all slides from all PDFs.
# - page_number is only metadata.
# - Gemini classifies slides based on content, not hardcoded page numbers.
# - Human review corrections are stored separately and can be used later.
#
# Table naming convention:
# sdi_competitiveOffers_{layer}_{purpose}_{weekly}
#
# Example:
# sdi_competitiveOffers_bronze_slideLines_weekly
# sdi_competitiveOffers_silver_normalizedOffers_weekly
# sdi_competitiveOffers_review_extractionReview_weekly
# ============================================================

print("=" * 100)
print("Notebook started: Competitive Offers PDF to Bronze, Silver, and Review")
print("=" * 100)

In [ ]:
# ============================================================
# 1. Install required packages
# ============================================================

print("\nSTEP 1: Installing required packages if needed...")

# In Colab Enterprise / Vertex Workbench, uncomment if packages are missing.
# !pip install google-cloud-bigquery google-genai pandas pyarrow pymupdf python-dateutil

print("STEP 1 completed: Package installation step is ready.")

In [ ]:
# ============================================================
# 2. Import libraries
# ============================================================

print("\nSTEP 2: Importing libraries...")

import os
import re
import json
import hashlib
from datetime import datetime, date
from dateutil import parser as date_parser

import pandas as pd
import fitz  # PyMuPDF

from google.cloud import bigquery

# Gemini / Vertex AI
from google import genai
from google.genai import types

print("STEP 2 completed: Libraries imported successfully.")

In [ ]:
# ============================================================
# 3. Configuration
# ============================================================

print("\nSTEP 3: Loading configuration...")

# ----------------------------
# Update these values
# ----------------------------
PROJECT_ID = "your-gcp-project-id"
BQ_DATASET = "your_bigquery_dataset"
LOCATION = "us-central1"

TEAM_NAME = "sdi"
PROJECT_NAME = "competitiveOffers"
CADENCE = "weekly"

# Gemini model
GEMINI_MODEL = "gemini-2.5-pro"

# Confidence thresholds
SECTION_CONFIDENCE_THRESHOLD = 0.80
NORMALIZATION_CONFIDENCE_THRESHOLD = 0.85

# During testing, set this to True if you want everything reviewed manually.
FORCE_ALL_ROWS_TO_REVIEW = False

# Auto approve clean rows?
AUTO_APPROVE_HIGH_CONFIDENCE = True

print(f"PROJECT_ID: {PROJECT_ID}")
print(f"BQ_DATASET: {BQ_DATASET}")
print(f"LOCATION: {LOCATION}")
print(f"GEMINI_MODEL: {GEMINI_MODEL}")
print(f"FORCE_ALL_ROWS_TO_REVIEW: {FORCE_ALL_ROWS_TO_REVIEW}")
print(f"AUTO_APPROVE_HIGH_CONFIDENCE: {AUTO_APPROVE_HIGH_CONFIDENCE}")

print("STEP 3 completed: Configuration loaded.")

In [ ]:
# ============================================================
# 4. Upload PDF manually inside notebook
# ============================================================

print("\nSTEP 4: Uploading PDF...")

# For MVP, we are NOT using GCS.
# GCS can be added later for production.
#
# Later production version:
# GCS_BUCKET = "your-bucket"
# GCS_PREFIX = "competitive_offers/raw_pdfs"
# gcs_pdf_path = f"gs://{GCS_BUCKET}/{GCS_PREFIX}/{pdf_name}"

PDF_LOCAL_PATH = None

try:
    from google.colab import files

    print("Colab-style upload detected.")
    print("Please upload one Competitive Offer Report PDF.")
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No file uploaded.")

    PDF_LOCAL_PATH = list(uploaded.keys())[0]
    print(f"Uploaded file: {PDF_LOCAL_PATH}")

except Exception as e:
    print("Colab upload widget was not used or is unavailable.")
    print("If you are using Colab Enterprise / Vertex Workbench, upload the PDF to the file browser.")
    print("Then manually set PDF_LOCAL_PATH below.")
    print(f"Upload widget message: {e}")

# Manual fallback
if PDF_LOCAL_PATH is None:
    PDF_LOCAL_PATH = "Competitive_Offer_Report_050826.pdf"

print(f"PDF_LOCAL_PATH is set to: {PDF_LOCAL_PATH}")

if not os.path.exists(PDF_LOCAL_PATH):
    raise FileNotFoundError(
        f"PDF not found at {PDF_LOCAL_PATH}. Upload the file or correct the path."
    )

print("STEP 4 completed: PDF path confirmed.")

In [ ]:
# ============================================================
# 5. Table naming convention
# ============================================================

print("\nSTEP 5: Creating table names...")

def build_table_name(layer: str, purpose: str, cadence: str = CADENCE) -> str:
    """
    Builds table names using the team convention.

    Format:
    sdi_competitiveOffers_{layer}_{purpose}_{weekly}

    Example:
    sdi_competitiveOffers_bronze_slideRaw_weekly
    """
    return f"{TEAM_NAME}_{PROJECT_NAME}_{layer}_{purpose}_{cadence}"


TABLES = {
    # Bronze
    "bronze_pdfDecks": build_table_name("bronze", "pdfDecks"),
    "bronze_slideRaw": build_table_name("bronze", "slideRaw"),
    "bronze_slideLines": build_table_name("bronze", "slideLines"),
    "bronze_detectedEntities": build_table_name("bronze", "detectedEntities"),
    "bronze_slideTables": build_table_name("bronze", "slideTables"),

    # Silver
    "silver_slideSections": build_table_name("silver", "slideSections"),
    "silver_offerCandidates": build_table_name("silver", "offerCandidates"),
    "silver_normalizedOffers": build_table_name("silver", "normalizedOffers"),
    "silver_offerDeviceBridge": build_table_name("silver", "offerDeviceBridge"),

    # Review
    "review_extractionReview": build_table_name("review", "extractionReview"),
    "review_reviewDecisions": build_table_name("review", "reviewDecisions"),
    "review_taxonomy": build_table_name("review", "taxonomy"),
    "review_approvedExamples": build_table_name("review", "approvedExamples"),

    # Gold, loaded in Notebook 2
    "gold_ciHeadlines": build_table_name("gold", "ciHeadlines"),
    "gold_majorOfferChanges": build_table_name("gold", "majorOfferChanges"),
    "gold_postpaidOffers": build_table_name("gold", "postpaidOffers"),
    "gold_businessOffers": build_table_name("gold", "businessOffers"),
    "gold_prepaidOffers": build_table_name("gold", "prepaidOffers"),
}

print("Tables that will be used:")
for key, value in TABLES.items():
    print(f"  {key}: {value}")

print("STEP 5 completed: Table names created.")

In [ ]:
# ============================================================
# 6. Initialize clients and helper functions
# ============================================================

print("\nSTEP 6: Initializing BigQuery and Gemini clients...")

bq_client = bigquery.Client(project=PROJECT_ID)

genai_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

print("Clients initialized.")

def full_table_id(table_short_name: str) -> str:
    return f"{PROJECT_ID}.{BQ_DATASET}.{table_short_name}"


def make_hash(value: str) -> str:
    value = value or ""
    return hashlib.md5(value.encode("utf-8")).hexdigest()


def get_file_hash(file_path: str) -> str:
    with open(file_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def clean_text(text: str) -> str:
    if text is None:
        return ""
    text = text.replace("\u00a0", " ")
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_line(text: str) -> str:
    if text is None:
        return ""
    text = text.replace("\u00a0", " ")
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    return text.strip()


def safe_json(value):
    return json.dumps(value, ensure_ascii=False, default=str)


def preview_df(df: pd.DataFrame, name: str, rows: int = 10):
    print("\n" + "=" * 100)
    print(f"DATAFRAME PREVIEW: {name}")
    print("=" * 100)
    print(f"Row count: {len(df)}")
    if len(df) > 0:
        display(df.head(rows))
    else:
        print("No rows found.")


def append_df_to_bq(df: pd.DataFrame, table_short_name: str):
    if df is None or df.empty:
        print(f"No rows to append to {table_short_name}. Skipping.")
        return

    table_id = full_table_id(table_short_name)

    print(f"Appending {len(df)} rows to {table_id}...")

    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
    job = bq_client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()

    print(f"Append completed: {len(df)} rows loaded into {table_id}.")


print("STEP 6 completed: Clients and helpers ready.")

In [ ]:
# ============================================================
# 7. Taxonomy and dictionaries
# ============================================================

print("\nSTEP 7: Defining taxonomy and dictionaries...")

ALLOWED_SECTION_NAMES = [
    "cover",
    "confidentiality_notice",
    "table_of_contents",
    "section_divider",
    "ci_headlines",
    "major_offer_changes",
    "postpaid_device_offers",
    "postpaid_bts_offers",
    "postpaid_service_offers",
    "cable_mvno_offers",
    "business_offers",
    "national_retail",
    "prepaid_offers",
    "unknown"
]

SECTION_KEYWORDS = {
    "confidentiality_notice": ["notice of confidentiality"],
    "table_of_contents": ["table of contents"],
    "ci_headlines": ["competitive intelligence headlines", "key highlights"],
    "major_offer_changes": ["major offer changes", "offers introduced", "offers removed"],
    "postpaid_device_offers": ["post-paid apple", "post-paid samsung", "smartphone offers", "byod offers"],
    "postpaid_bts_offers": ["post-paid bts", "watches", "tablet", "connected devices"],
    "postpaid_service_offers": ["post-paid service offers", "byod wireless service", "value-added services"],
    "cable_mvno_offers": ["cable mvno", "optimum mobile", "spectrum mobile", "xfinity mobile"],
    "business_offers": ["business", "flagship device offers", "my biz", "supermobile", "promobile", "coremobile"],
    "national_retail": ["best buy", "walmart", "weekly readout", "national retail"],
    "prepaid_offers": ["prepaid", "metro", "boost mobile", "cricket", "straight talk", "total wireless"],
    "section_divider": ["competitive offers & promotional updates", "ci headlines"]
}

CARRIER_KEYWORDS = {
    "T-Mobile": ["t-mobile", "tmobile"],
    "Metro by T-Mobile": ["metro by t-mobile", "metro"],
    "Verizon": ["verizon"],
    "AT&T": ["at&t", "att"],
    "Spectrum": ["spectrum"],
    "Xfinity": ["xfinity", "comcast"],
    "Optimum Mobile": ["optimum"],
    "Boost Mobile": ["boost"],
    "Cricket Wireless": ["cricket"],
    "Straight Talk": ["straight talk"],
    "Total Wireless": ["total wireless"],
    "Visible": ["visible"],
    "Tello": ["tello"],
    "Google Fi": ["google fi"],
    "Simple Mobile": ["simple mobile"]
}

MANUFACTURER_KEYWORDS = {
    "Apple": ["iphone", "ipad", "apple watch"],
    "Samsung": ["samsung", "galaxy", "z flip", "z fold", "tab"],
    "Google": ["pixel"],
    "Motorola": ["moto", "motorola", "razr"],
    "TCL": ["tcl"],
    "Revvl": ["revvl"]
}

PRODUCT_FAMILY_RULES = {
    "smartphone": ["iphone", "galaxy", "pixel", "moto", "razr", "phone", "handset"],
    "watch": ["watch", "apple watch", "galaxy watch", "pixel watch", "gizmo"],
    "tablet": ["tablet", "ipad", "tab"],
    "byod": ["byod", "bring your own"],
    "service_plan": ["plan", "unlimited", "rate plan", "service"],
    "home_internet": ["home internet", "fios", "fiber", "hint", "gateway"],
    "hotspot": ["hotspot", "mifi", "jetpack"]
}

OFFER_SUBCATEGORY_RULES = {
    "device_discount": ["off", "discount", "save", "savings"],
    "trade_in_offer": ["trade", "trade-in", "trd", "fmv", "any condition"],
    "free_device_offer": ["free", "on us", "$0/mo", "$0"],
    "monthly_bill_credit": ["bill credit", "monthly credit", "credits over", "x 36", "x 24"],
    "byod_credit": ["byod", "bring your own"],
    "eip_buyout": ["eip buyout", "pay off", "payoff", "switcher"],
    "port_in_credit": ["port-in credit", "port credit"],
    "aal_offer": ["aal", "add a line", "new line"],
    "plan_refresh": ["plan refresh", "new plan", "updated existing plans", "rate plan"],
    "price_increase": ["price increase", "+$", "higher than prior"],
    "activation_fee_waiver": ["waived activation", "activation fee waived", "waived fee"],
    "expired_offer": ["expired", "removed", "retired", "ended"],
    "no_change": ["no changes", "no updates", "no offers ended", "no new offers"]
}

def dictionary_match(text: str, dictionary: dict):
    text_lower = (text or "").lower()
    scores = {}

    for label, keywords in dictionary.items():
        matched = [kw for kw in keywords if kw.lower() in text_lower]
        if matched:
            scores[label] = {
                "score": len(matched),
                "matched_keywords": matched
            }

    if not scores:
        return None, 0.0, []

    best_label = max(scores, key=lambda x: scores[x]["score"])
    best_score = scores[best_label]["score"]
    confidence = min(best_score / 3, 1.0)

    return best_label, confidence, scores[best_label]["matched_keywords"]


def infer_offer_category(offer_subcategory: str):
    if offer_subcategory in ["device_discount", "trade_in_offer", "free_device_offer"]:
        return "device_promotion"
    if offer_subcategory in ["monthly_bill_credit", "byod_credit", "eip_buyout", "port_in_credit", "aal_offer"]:
        return "service_promotion"
    if offer_subcategory in ["plan_refresh", "price_increase"]:
        return "plan_pricing"
    if offer_subcategory in ["activation_fee_waiver"]:
        return "fee_promotion"
    if offer_subcategory in ["expired_offer", "no_change"]:
        return "offer_status"
    return "unknown"

print("STEP 7 completed: Taxonomy and dictionaries ready.")

In [ ]:
# ============================================================
# 8. Entity extraction helper functions
# ============================================================

print("\nSTEP 8: Defining extraction helper functions...")

def detect_money_values(text: str):
    pattern = r"(?:<|>|~)?\$\s?\d{1,3}(?:,\d{3})*(?:\.\d{1,2})?"
    return re.findall(pattern, text or "")


def normalize_money_value(money_text: str):
    if not money_text:
        return None
    value = (
        money_text
        .replace("$", "")
        .replace(",", "")
        .replace("<", "")
        .replace(">", "")
        .replace("~", "")
        .strip()
    )
    try:
        return float(value)
    except Exception:
        return None


def detect_dates(text: str):
    pattern = r"\d{1,2}/\d{1,2}(?:\s?-\s?(?:\d{1,2}/\d{1,2}|TBD)?)?"
    return re.findall(pattern, text or "", flags=re.IGNORECASE)


def parse_date_range(date_text: str, deck_year: int):
    if not date_text:
        return None, None

    cleaned = date_text.replace("–", "-").replace("—", "-")
    match = re.search(
        r"(\d{1,2}/\d{1,2})(?:\s?-\s?(TBD|\d{1,2}/\d{1,2})?)?",
        cleaned,
        flags=re.IGNORECASE
    )

    if not match:
        return None, None

    start_raw = match.group(1)
    end_raw = match.group(2)

    start_date = None
    end_date = None

    try:
        start_date = date_parser.parse(f"{start_raw}/{deck_year}").date().isoformat()
    except Exception:
        pass

    if end_raw and end_raw.upper() != "TBD":
        try:
            end_date = date_parser.parse(f"{end_raw}/{deck_year}").date().isoformat()
        except Exception:
            pass

    return start_date, end_date


def detect_data_allowance(text: str):
    return re.findall(r"\b\d+\s?GB\b", text or "", flags=re.IGNORECASE)


def detect_term_months(text: str):
    matches = re.findall(r"\b\d{1,2}\s?(?:months|mos|mo)\b", text or "", flags=re.IGNORECASE)
    if not matches:
        return None
    num_match = re.search(r"\d{1,2}", matches[0])
    return int(num_match.group(0)) if num_match else None


def detect_carrier(text: str):
    return dictionary_match(text, CARRIER_KEYWORDS)


def detect_manufacturer(text: str):
    return dictionary_match(text, MANUFACTURER_KEYWORDS)


def detect_product_family(text: str):
    return dictionary_match(text, PRODUCT_FAMILY_RULES)


def detect_offer_subcategory(text: str):
    return dictionary_match(text, OFFER_SUBCATEGORY_RULES)


def extract_device_models(text: str):
    """
    Extracts one or many devices from the text.
    This supports grouped device families.
    Example:
    - iPhone 16
    - iPhone 17, iPhone Air, iPhone 17 Pro, iPhone 17 Pro Max
    """
    if not text:
        return []

    patterns = [
        r"iPhone\s?\d{1,2}[a-zA-Z]?(?:\s?Pro Max|\s?Pro|\s?Plus|\s?Air|\s?e)?",
        r"iPhone\s?Air",
        r"Galaxy\s?S\d{2}(?:\s?FE|\s?Ultra|\s?Edge|\+)?",
        r"Galaxy\s?A\d{2}\s?5G",
        r"Galaxy\s?Z\s?(?:Flip|Fold)\d+",
        r"Z\s?(?:Flip|Fold)\d+",
        r"Pixel\s?\d{1,2}[a-zA-Z]?(?:\s?Pro XL|\s?Pro|\s?Fold)?",
        r"moto\s?g\s?\w*(?:\s?5G)?(?:\s?2025|\s?2026)?",
        r"moto\s?edge\s?\d{4}",
        r"moto\s?razr\s?\d{4}",
        r"Razr\+?\s?(?:Ultra)?",
        r"Apple Watch\s?[A-Za-z0-9 ]*",
        r"Galaxy Watch\d*",
        r"iPad\s?[A-Za-z0-9 ]*",
        r"Galaxy Tab\s?[A-Za-z0-9+ ]*",
    ]

    found = []

    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        for match in matches:
            cleaned = clean_text(match)
            if cleaned and cleaned.lower() not in [x.lower() for x in found]:
                found.append(cleaned)

    return found

print("STEP 8 completed: Extraction helper functions ready.")

In [ ]:
# ============================================================
# 9. Define schemas and create tables
# ============================================================

print("\nSTEP 9: Defining schemas and creating BigQuery tables...")

COMMON_KEYS = [
    bigquery.SchemaField("deck_id", "STRING"),
    bigquery.SchemaField("run_id", "STRING"),
    bigquery.SchemaField("pdf_hash", "STRING"),
    bigquery.SchemaField("pdf_name", "STRING"),
    bigquery.SchemaField("deck_week", "DATE"),
]

SCHEMA_BRONZE_PDF_DECKS = [
    bigquery.SchemaField("deck_id", "STRING"),
    bigquery.SchemaField("run_id", "STRING"),
    bigquery.SchemaField("pdf_hash", "STRING"),
    bigquery.SchemaField("pdf_name", "STRING"),
    bigquery.SchemaField("deck_week", "DATE"),
    bigquery.SchemaField("local_pdf_path", "STRING"),
    bigquery.SchemaField("total_pages", "INTEGER"),
    bigquery.SchemaField("processing_mode", "STRING"),
    bigquery.SchemaField("processing_status", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_BRONZE_SLIDE_RAW = COMMON_KEYS + [
    bigquery.SchemaField("slide_id", "STRING"),
    bigquery.SchemaField("page_number", "INTEGER"),
    bigquery.SchemaField("raw_slide_text", "STRING"),
    bigquery.SchemaField("cleaned_slide_text", "STRING"),
    bigquery.SchemaField("slide_text_hash", "STRING"),
    bigquery.SchemaField("extraction_method", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_BRONZE_SLIDE_LINES = COMMON_KEYS + [
    bigquery.SchemaField("slide_id", "STRING"),
    bigquery.SchemaField("line_id", "STRING"),
    bigquery.SchemaField("page_number", "INTEGER"),
    bigquery.SchemaField("line_number_on_page", "INTEGER"),
    bigquery.SchemaField("raw_line_text", "STRING"),
    bigquery.SchemaField("cleaned_line_text", "STRING"),
    bigquery.SchemaField("line_text_hash", "STRING"),
    bigquery.SchemaField("line_type", "STRING"),
    bigquery.SchemaField("source_method", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_BRONZE_DETECTED_ENTITIES = COMMON_KEYS + [
    bigquery.SchemaField("entity_id", "STRING"),
    bigquery.SchemaField("slide_id", "STRING"),
    bigquery.SchemaField("line_id", "STRING"),
    bigquery.SchemaField("page_number", "INTEGER"),
    bigquery.SchemaField("entity_type", "STRING"),
    bigquery.SchemaField("entity_text", "STRING"),
    bigquery.SchemaField("normalized_entity_value", "STRING"),
    bigquery.SchemaField("detection_method", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_BRONZE_SLIDE_TABLES = COMMON_KEYS + [
    bigquery.SchemaField("slide_table_id", "STRING"),
    bigquery.SchemaField("slide_id", "STRING"),
    bigquery.SchemaField("page_number", "INTEGER"),
    bigquery.SchemaField("table_json", "STRING"),
    bigquery.SchemaField("extraction_method", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_SILVER_SLIDE_SECTIONS = COMMON_KEYS + [
    bigquery.SchemaField("slide_id", "STRING"),
    bigquery.SchemaField("page_number", "INTEGER"),
    bigquery.SchemaField("slide_title", "STRING"),
    bigquery.SchemaField("page_type", "STRING"),
    bigquery.SchemaField("predicted_section_name", "STRING"),
    bigquery.SchemaField("business_segment", "STRING"),
    bigquery.SchemaField("section_confidence", "FLOAT"),
    bigquery.SchemaField("section_reason", "STRING"),
    bigquery.SchemaField("classification_source", "STRING"),
    bigquery.SchemaField("review_required_flag", "BOOLEAN"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_SILVER_OFFER_CANDIDATES = COMMON_KEYS + [
    bigquery.SchemaField("candidate_id", "STRING"),
    bigquery.SchemaField("offer_group_id", "STRING"),
    bigquery.SchemaField("slide_id", "STRING"),
    bigquery.SchemaField("line_id", "STRING"),
    bigquery.SchemaField("page_number", "INTEGER"),
    bigquery.SchemaField("section_name", "STRING"),
    bigquery.SchemaField("business_segment", "STRING"),
    bigquery.SchemaField("carrier", "STRING"),
    bigquery.SchemaField("raw_offer_text", "STRING"),
    bigquery.SchemaField("context_before_text", "STRING"),
    bigquery.SchemaField("context_after_text", "STRING"),
    bigquery.SchemaField("suggested_offer_category", "STRING"),
    bigquery.SchemaField("suggested_offer_subcategory", "STRING"),
    bigquery.SchemaField("detected_money_values_json", "STRING"),
    bigquery.SchemaField("detected_device_models_json", "STRING"),
    bigquery.SchemaField("detected_dates_json", "STRING"),
    bigquery.SchemaField("candidate_confidence", "FLOAT"),
    bigquery.SchemaField("candidate_source", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_SILVER_NORMALIZED_OFFERS = COMMON_KEYS + [
    bigquery.SchemaField("offer_id", "STRING"),
    bigquery.SchemaField("offer_group_id", "STRING"),
    bigquery.SchemaField("candidate_id", "STRING"),
    bigquery.SchemaField("slide_id", "STRING"),
    bigquery.SchemaField("line_id", "STRING"),
    bigquery.SchemaField("page_number", "INTEGER"),
    bigquery.SchemaField("section_name", "STRING"),
    bigquery.SchemaField("business_segment", "STRING"),
    bigquery.SchemaField("carrier", "STRING"),
    bigquery.SchemaField("customer_segment", "STRING"),
    bigquery.SchemaField("product_family", "STRING"),
    bigquery.SchemaField("manufacturer", "STRING"),
    bigquery.SchemaField("device_family_text", "STRING"),
    bigquery.SchemaField("primary_device_model", "STRING"),
    bigquery.SchemaField("device_models_json", "STRING"),
    bigquery.SchemaField("plan_name", "STRING"),
    bigquery.SchemaField("offer_category", "STRING"),
    bigquery.SchemaField("offer_subcategory", "STRING"),
    bigquery.SchemaField("offer_type", "STRING"),
    bigquery.SchemaField("offer_value", "FLOAT"),
    bigquery.SchemaField("offer_value_unit", "STRING"),
    bigquery.SchemaField("monthly_price", "FLOAT"),
    bigquery.SchemaField("data_allowance", "STRING"),
    bigquery.SchemaField("term_length_months", "INTEGER"),
    bigquery.SchemaField("customer_action", "STRING"),
    bigquery.SchemaField("condition_text", "STRING"),
    bigquery.SchemaField("port_required_flag", "BOOLEAN"),
    bigquery.SchemaField("aal_required_flag", "BOOLEAN"),
    bigquery.SchemaField("trade_required_flag", "BOOLEAN"),
    bigquery.SchemaField("upgrade_required_flag", "BOOLEAN"),
    bigquery.SchemaField("new_line_required_flag", "BOOLEAN"),
    bigquery.SchemaField("online_only_flag", "BOOLEAN"),
    bigquery.SchemaField("in_store_only_flag", "BOOLEAN"),
    bigquery.SchemaField("is_free_flag", "BOOLEAN"),
    bigquery.SchemaField("is_new_flag", "BOOLEAN"),
    bigquery.SchemaField("is_expired_flag", "BOOLEAN"),
    bigquery.SchemaField("effective_start_date", "DATE"),
    bigquery.SchemaField("effective_end_date", "DATE"),
    bigquery.SchemaField("source_raw_text", "STRING"),
    bigquery.SchemaField("category_confidence", "FLOAT"),
    bigquery.SchemaField("normalization_confidence", "FLOAT"),
    bigquery.SchemaField("validation_status", "STRING"),
    bigquery.SchemaField("review_required_flag", "BOOLEAN"),
    bigquery.SchemaField("review_status", "STRING"),
    bigquery.SchemaField("status", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_SILVER_OFFER_DEVICE_BRIDGE = COMMON_KEYS + [
    bigquery.SchemaField("offer_device_id", "STRING"),
    bigquery.SchemaField("offer_id", "STRING"),
    bigquery.SchemaField("offer_group_id", "STRING"),
    bigquery.SchemaField("page_number", "INTEGER"),
    bigquery.SchemaField("manufacturer", "STRING"),
    bigquery.SchemaField("device_model", "STRING"),
    bigquery.SchemaField("device_rank", "INTEGER"),
    bigquery.SchemaField("source_raw_text", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_REVIEW_EXTRACTION_REVIEW = COMMON_KEYS + [
    bigquery.SchemaField("review_id", "STRING"),
    bigquery.SchemaField("offer_id", "STRING"),
    bigquery.SchemaField("offer_group_id", "STRING"),
    bigquery.SchemaField("candidate_id", "STRING"),
    bigquery.SchemaField("page_number", "INTEGER"),
    bigquery.SchemaField("section_name", "STRING"),
    bigquery.SchemaField("business_segment", "STRING"),
    bigquery.SchemaField("carrier", "STRING"),
    bigquery.SchemaField("source_raw_text", "STRING"),
    bigquery.SchemaField("suggested_output_json", "STRING"),
    bigquery.SchemaField("confidence_score", "FLOAT"),
    bigquery.SchemaField("validation_status", "STRING"),
    bigquery.SchemaField("review_reason", "STRING"),
    bigquery.SchemaField("review_status", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_REVIEW_DECISIONS = COMMON_KEYS + [
    bigquery.SchemaField("decision_id", "STRING"),
    bigquery.SchemaField("review_id", "STRING"),
    bigquery.SchemaField("offer_id", "STRING"),
    bigquery.SchemaField("offer_group_id", "STRING"),
    bigquery.SchemaField("decision_type", "STRING"),
    bigquery.SchemaField("review_status", "STRING"),
    bigquery.SchemaField("reviewer_name", "STRING"),
    bigquery.SchemaField("review_notes", "STRING"),
    bigquery.SchemaField("final_output_json", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_REVIEW_TAXONOMY = COMMON_KEYS + [
    bigquery.SchemaField("taxonomy_id", "STRING"),
    bigquery.SchemaField("taxonomy_type", "STRING"),
    bigquery.SchemaField("taxonomy_value", "STRING"),
    bigquery.SchemaField("taxonomy_description", "STRING"),
    bigquery.SchemaField("example_text", "STRING"),
    bigquery.SchemaField("active_flag", "BOOLEAN"),
    bigquery.SchemaField("created_by", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

SCHEMA_REVIEW_APPROVED_EXAMPLES = COMMON_KEYS + [
    bigquery.SchemaField("example_id", "STRING"),
    bigquery.SchemaField("source_raw_text", "STRING"),
    bigquery.SchemaField("approved_output_json", "STRING"),
    bigquery.SchemaField("approved_by", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]


def create_table_if_not_exists(table_short_name: str, schema: list):
    table_id = full_table_id(table_short_name)

    try:
        bq_client.get_table(table_id)
        print(f"Table already exists: {table_id}")
        return
    except Exception:
        print(f"Creating table: {table_id}")

    table = bigquery.Table(table_id, schema=schema)
    bq_client.create_table(table)
    print(f"Created table: {table_id}")


create_table_if_not_exists(TABLES["bronze_pdfDecks"], SCHEMA_BRONZE_PDF_DECKS)
create_table_if_not_exists(TABLES["bronze_slideRaw"], SCHEMA_BRONZE_SLIDE_RAW)
create_table_if_not_exists(TABLES["bronze_slideLines"], SCHEMA_BRONZE_SLIDE_LINES)
create_table_if_not_exists(TABLES["bronze_detectedEntities"], SCHEMA_BRONZE_DETECTED_ENTITIES)
create_table_if_not_exists(TABLES["bronze_slideTables"], SCHEMA_BRONZE_SLIDE_TABLES)

create_table_if_not_exists(TABLES["silver_slideSections"], SCHEMA_SILVER_SLIDE_SECTIONS)
create_table_if_not_exists(TABLES["silver_offerCandidates"], SCHEMA_SILVER_OFFER_CANDIDATES)
create_table_if_not_exists(TABLES["silver_normalizedOffers"], SCHEMA_SILVER_NORMALIZED_OFFERS)
create_table_if_not_exists(TABLES["silver_offerDeviceBridge"], SCHEMA_SILVER_OFFER_DEVICE_BRIDGE)

create_table_if_not_exists(TABLES["review_extractionReview"], SCHEMA_REVIEW_EXTRACTION_REVIEW)
create_table_if_not_exists(TABLES["review_reviewDecisions"], SCHEMA_REVIEW_DECISIONS)
create_table_if_not_exists(TABLES["review_taxonomy"], SCHEMA_REVIEW_TAXONOMY)
create_table_if_not_exists(TABLES["review_approvedExamples"], SCHEMA_REVIEW_APPROVED_EXAMPLES)

print("STEP 9 completed: All required tables exist.")

In [ ]:
# ============================================================
# 10. Register PDF and handle duplicate uploads
# ============================================================

print("\nSTEP 10: Reading PDF metadata and checking duplicates...")

pdf_name = os.path.basename(PDF_LOCAL_PATH)
pdf_hash = get_file_hash(PDF_LOCAL_PATH)

date_match = re.search(r"(\d{2})(\d{2})(\d{2})", pdf_name)

if date_match:
    month = int(date_match.group(1))
    day = int(date_match.group(2))
    year = int("20" + date_match.group(3))
    deck_week = date(year, month, day).isoformat()
else:
    year = date.today().year
    deck_week = date.today().isoformat()

deck_id = make_hash(f"{pdf_hash}|{deck_week}")
run_id = make_hash(f"{deck_id}|{datetime.utcnow().isoformat()}")

doc = fitz.open(PDF_LOCAL_PATH)
total_pages = len(doc)
doc.close()

print(f"PDF name: {pdf_name}")
print(f"PDF hash: {pdf_hash}")
print(f"Deck week: {deck_week}")
print(f"Deck year: {year}")
print(f"Deck ID: {deck_id}")
print(f"Run ID: {run_id}")
print(f"Total pages: {total_pages}")

def check_existing_pdf(pdf_hash: str) -> pd.DataFrame:
    table_id = full_table_id(TABLES["bronze_pdfDecks"])

    query = f"""
    SELECT
      deck_id,
      run_id,
      pdf_hash,
      pdf_name,
      deck_week,
      processing_status,
      created_at
    FROM `{table_id}`
    WHERE pdf_hash = '{pdf_hash}'
    ORDER BY created_at DESC
    """

    try:
        return bq_client.query(query).to_dataframe()
    except Exception as e:
        print("Could not check existing PDF.")
        print(e)
        return pd.DataFrame()


existing_pdf_df = check_existing_pdf(pdf_hash)

if existing_pdf_df.empty:
    print("This PDF does not exist in BigQuery yet. Processing mode = new_append.")
    PDF_PROCESSING_MODE = "new_append"
else:
    print("This exact PDF already exists in BigQuery.")
    preview_df(existing_pdf_df, "Existing PDF records")

    print("""
Choose processing mode:

cancel
  Stop the notebook and do nothing.

replace
  Delete existing rows for this deck_id/pdf_hash and reprocess the PDF.

append_anyway
  Append this as another run_id even though the PDF already exists.
  Useful for debugging only.
""")

    PDF_PROCESSING_MODE = input("Enter processing mode [cancel / replace / append_anyway]: ").strip().lower()

    if PDF_PROCESSING_MODE not in ["cancel", "replace", "append_anyway"]:
        raise ValueError("Invalid processing mode.")

print(f"Selected PDF_PROCESSING_MODE: {PDF_PROCESSING_MODE}")

if PDF_PROCESSING_MODE == "cancel":
    print("User selected cancel. No data changed.")
    raise SystemExit("Notebook stopped because user selected cancel.")


def delete_existing_rows_for_pdf(deck_id: str, pdf_hash: str):
    print("Deleting existing rows for replacement...")

    tables_with_deck_id = [
        TABLES["bronze_slideRaw"],
        TABLES["bronze_slideLines"],
        TABLES["bronze_detectedEntities"],
        TABLES["bronze_slideTables"],
        TABLES["silver_slideSections"],
        TABLES["silver_offerCandidates"],
        TABLES["silver_normalizedOffers"],
        TABLES["silver_offerDeviceBridge"],
        TABLES["review_extractionReview"],
        TABLES["review_reviewDecisions"],
        TABLES["review_taxonomy"],
        TABLES["review_approvedExamples"],
        TABLES["gold_ciHeadlines"],
        TABLES["gold_majorOfferChanges"],
        TABLES["gold_postpaidOffers"],
        TABLES["gold_businessOffers"],
        TABLES["gold_prepaidOffers"],
    ]

    for table_short_name in tables_with_deck_id:
        table_id = full_table_id(table_short_name)
        query = f"DELETE FROM `{table_id}` WHERE deck_id = '{deck_id}'"

        try:
            print(f"Deleting from {table_id}...")
            bq_client.query(query).result()
            print("Deleted matching rows.")
        except Exception as e:
            print(f"Could not delete from {table_id}. It may not exist yet or may not have deck_id.")
            print(e)

    pdf_decks_table = full_table_id(TABLES["bronze_pdfDecks"])
    query = f"DELETE FROM `{pdf_decks_table}` WHERE pdf_hash = '{pdf_hash}'"

    print(f"Deleting PDF registration from {pdf_decks_table}...")
    bq_client.query(query).result()
    print("PDF registration deleted.")

if PDF_PROCESSING_MODE == "replace":
    delete_existing_rows_for_pdf(deck_id, pdf_hash)
    print("Replacement cleanup completed. Notebook will now append fresh rows.")

print("STEP 10 completed: Duplicate handling done.")

In [ ]:
# ============================================================
# 11. Insert PDF deck record
# ============================================================

print("\nSTEP 11: Inserting PDF deck record...")

bronze_pdf_decks_df = pd.DataFrame([{
    "deck_id": deck_id,
    "run_id": run_id,
    "pdf_hash": pdf_hash,
    "pdf_name": pdf_name,
    "deck_week": deck_week,
    "local_pdf_path": PDF_LOCAL_PATH,
    "total_pages": total_pages,
    "processing_mode": PDF_PROCESSING_MODE,
    "processing_status": "registered",
    "created_at": datetime.utcnow()
}])

preview_df(bronze_pdf_decks_df, "bronze_pdf_decks_df")

append_df_to_bq(bronze_pdf_decks_df, TABLES["bronze_pdfDecks"])

print("STEP 11 completed: PDF deck record inserted.")

In [ ]:
# ============================================================
# 12. Extract bronze slide raw and slide lines
# ============================================================

print("\nSTEP 12: Extracting raw slide text and slide lines...")

def extract_pdf_to_bronze(pdf_path: str):
    doc = fitz.open(pdf_path)

    slide_raw_records = []
    slide_line_records = []

    for page_index in range(len(doc)):
        page_number = page_index + 1
        page = doc[page_index]

        print(f"Extracting page {page_number}/{len(doc)}...")

        raw_text = page.get_text("text") or ""
        cleaned_slide_text = clean_text(raw_text)

        slide_id = make_hash(f"{deck_id}|page|{page_number}")

        slide_raw_records.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "slide_id": slide_id,
            "page_number": page_number,
            "raw_slide_text": raw_text,
            "cleaned_slide_text": cleaned_slide_text,
            "slide_text_hash": make_hash(cleaned_slide_text),
            "extraction_method": "pymupdf_text",
            "created_at": datetime.utcnow()
        })

        for line_number, raw_line in enumerate(raw_text.splitlines(), start=1):
            cleaned = clean_line(raw_line)

            if not cleaned:
                continue

            line_id = make_hash(f"{slide_id}|line|{line_number}|{cleaned}")

            slide_line_records.append({
                "deck_id": deck_id,
                "run_id": run_id,
                "pdf_hash": pdf_hash,
                "pdf_name": pdf_name,
                "deck_week": deck_week,
                "slide_id": slide_id,
                "line_id": line_id,
                "page_number": page_number,
                "line_number_on_page": line_number,
                "raw_line_text": raw_line,
                "cleaned_line_text": cleaned,
                "line_text_hash": make_hash(cleaned),
                "line_type": "unknown",
                "source_method": "pymupdf_text",
                "created_at": datetime.utcnow()
            })

    doc.close()

    return pd.DataFrame(slide_raw_records), pd.DataFrame(slide_line_records)


bronze_slide_raw_df, bronze_slide_lines_df = extract_pdf_to_bronze(PDF_LOCAL_PATH)

preview_df(bronze_slide_raw_df, "bronze_slide_raw_df")
preview_df(bronze_slide_lines_df, "bronze_slide_lines_df", rows=20)

append_df_to_bq(bronze_slide_raw_df, TABLES["bronze_slideRaw"])
append_df_to_bq(bronze_slide_lines_df, TABLES["bronze_slideLines"])

print("STEP 12 completed: Bronze slide raw and slide lines loaded.")

In [ ]:
# ============================================================
# 13. Extract detected entities
# ============================================================

print("\nSTEP 13: Extracting detected entities...")

def create_detected_entities(slide_lines_df: pd.DataFrame):
    records = []

    for _, row in slide_lines_df.iterrows():
        text = row["cleaned_line_text"]

        detected_items = []

        for money in detect_money_values(text):
            detected_items.append(("money", money, str(normalize_money_value(money))))

        for dt in detect_dates(text):
            detected_items.append(("date_range", dt, dt))

        for data in detect_data_allowance(text):
            detected_items.append(("data_allowance", data, data.upper().replace(" ", "")))

        for device in extract_device_models(text):
            detected_items.append(("device_model", device, device))

        carrier, _, _ = detect_carrier(text)
        if carrier:
            detected_items.append(("carrier", carrier, carrier))

        for entity_type, entity_text, normalized_value in detected_items:
            records.append({
                "deck_id": deck_id,
                "run_id": run_id,
                "pdf_hash": pdf_hash,
                "pdf_name": pdf_name,
                "deck_week": deck_week,
                "entity_id": make_hash(f"{row['line_id']}|{entity_type}|{entity_text}"),
                "slide_id": row["slide_id"],
                "line_id": row["line_id"],
                "page_number": row["page_number"],
                "entity_type": entity_type,
                "entity_text": entity_text,
                "normalized_entity_value": normalized_value,
                "detection_method": "regex_dictionary",
                "created_at": datetime.utcnow()
            })

    return pd.DataFrame(records)


bronze_detected_entities_df = create_detected_entities(bronze_slide_lines_df)

preview_df(bronze_detected_entities_df, "bronze_detected_entities_df", rows=20)

append_df_to_bq(bronze_detected_entities_df, TABLES["bronze_detectedEntities"])

print("STEP 13 completed: Detected entities loaded.")

In [ ]:
# ============================================================
# 14. Classify slides into sections
# ============================================================

print("\nSTEP 14: Classifying slides into sections...")

def infer_business_segment(section_name: str, slide_text: str):
    text = (slide_text or "").lower()

    if section_name == "business_offers" or "business" in text:
        return "business"
    if section_name in ["prepaid_offers", "national_retail"] or "prepaid" in text:
        return "prepaid"
    if section_name in ["postpaid_device_offers", "postpaid_bts_offers", "postpaid_service_offers"]:
        return "postpaid"
    if section_name == "cable_mvno_offers":
        return "cable_mvno"
    if section_name in ["major_offer_changes", "ci_headlines"]:
        return "mixed"

    return "unknown"


def extract_slide_title(slide_text: str):
    if not slide_text:
        return None

    known_titles = [
        "Competitive Intelligence Headlines",
        "Major Offer Changes",
        "Post-Paid Apple Smartphone Offers",
        "Post-Paid Samsung and Google Smartphone Offers",
        "Post-Paid Motorola",
        "Post-Paid BTS",
        "T-Mobile & AT&T Post-Paid Service Offers",
        "Verizon Post-Paid Service Offers",
        "Postpaid Cable MVNO",
        "Cable MVNO Service",
        "Business",
        "Prepaid Brands",
        "Flagship Brands",
        "Walmart"
    ]

    lower = slide_text.lower()
    for title in known_titles:
        if title.lower() in lower:
            return title

    return slide_text[:150]


def classify_slide_with_dictionary(slide_text: str):
    section, confidence, matched = dictionary_match(slide_text, SECTION_KEYWORDS)

    if not section:
        return {
            "page_type": "unknown",
            "predicted_section_name": "unknown",
            "business_segment": "unknown",
            "section_confidence": 0.0,
            "section_reason": "No dictionary match found.",
            "classification_source": "dictionary",
            "review_required_flag": True
        }

    return {
        "page_type": "content_slide",
        "predicted_section_name": section,
        "business_segment": infer_business_segment(section, slide_text),
        "section_confidence": confidence,
        "section_reason": f"Matched keywords: {matched}",
        "classification_source": "dictionary",
        "review_required_flag": confidence < SECTION_CONFIDENCE_THRESHOLD
    }


def classify_slide_with_gemini(slide_text: str):
    prompt = f"""
You are classifying one slide/page from a telecom competitive offer report.

Classify the slide using content only:
- slide title
- section headers
- carrier names
- device names
- plan names
- offer language
- table headers

Do NOT use page number as the source of truth.

Allowed section names:
{ALLOWED_SECTION_NAMES}

Return JSON only with this schema:
{{
  "page_type": "cover | confidentiality_notice | table_of_contents | section_divider | content_slide | unknown",
  "predicted_section_name": "one allowed section name",
  "business_segment": "postpaid | prepaid | business | cable_mvno | national_retail | mixed | unknown",
  "section_confidence": 0.0,
  "section_reason": "short explanation",
  "review_required_flag": true
}}

Slide text:
\"\"\"
{slide_text[:12000]}
\"\"\"
"""

    try:
        response = genai_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json"
            )
        )

        return json.loads(response.text)

    except Exception as e:
        print(f"Gemini classification failed: {e}")
        return {
            "page_type": "unknown",
            "predicted_section_name": "unknown",
            "business_segment": "unknown",
            "section_confidence": 0.0,
            "section_reason": f"Gemini failed: {e}",
            "review_required_flag": True
        }


def classify_all_slides(slide_raw_df: pd.DataFrame):
    records = []

    for _, row in slide_raw_df.iterrows():
        page_number = row["page_number"]
        slide_text = row["cleaned_slide_text"]

        print(f"Classifying page {page_number}...")

        dict_result = classify_slide_with_dictionary(slide_text)

        if dict_result["section_confidence"] >= SECTION_CONFIDENCE_THRESHOLD:
            result = dict_result
            print(f"Page {page_number}: dictionary accepted as {result['predicted_section_name']}")
        else:
            print(f"Page {page_number}: dictionary low confidence. Calling Gemini...")
            gemini_result = classify_slide_with_gemini(slide_text)

            result = {
                "page_type": gemini_result.get("page_type", "unknown"),
                "predicted_section_name": gemini_result.get("predicted_section_name", "unknown"),
                "business_segment": gemini_result.get("business_segment", "unknown"),
                "section_confidence": float(gemini_result.get("section_confidence", 0.0)),
                "section_reason": gemini_result.get("section_reason", ""),
                "classification_source": "gemini",
                "review_required_flag": bool(gemini_result.get("review_required_flag", True)),
            }

            print(f"Page {page_number}: Gemini classified as {result['predicted_section_name']}")

        records.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "slide_id": row["slide_id"],
            "page_number": page_number,
            "slide_title": extract_slide_title(slide_text),
            "page_type": result["page_type"],
            "predicted_section_name": result["predicted_section_name"],
            "business_segment": result["business_segment"],
            "section_confidence": result["section_confidence"],
            "section_reason": result["section_reason"],
            "classification_source": result["classification_source"],
            "review_required_flag": result["review_required_flag"],
            "created_at": datetime.utcnow()
        })

    return pd.DataFrame(records)


silver_slide_sections_df = classify_all_slides(bronze_slide_raw_df)

preview_df(silver_slide_sections_df, "silver_slide_sections_df", rows=30)

append_df_to_bq(silver_slide_sections_df, TABLES["silver_slideSections"])

print("STEP 14 completed: Slide classification loaded.")

In [ ]:
# ============================================================
# 15. Create silver offer candidates
# ============================================================

print("\nSTEP 15: Creating offer candidates...")

def is_offer_like_line(text: str):
    if not text:
        return False

    lower = text.lower()

    signals = [
        "$", "off", "free", "on us", "credit", "trade", "port", "aal",
        "byod", "plan", "new", "expired", "removed", "retired",
        "no changes", "no updates", "price increase", "discount",
        "bogo", "waived", "activation"
    ]

    return any(signal in lower for signal in signals)


def infer_carrier_from_context(current_text: str, previous_lines: list):
    carrier, conf, matched = detect_carrier(current_text)
    if carrier:
        return carrier, conf, f"current_line:{matched}"

    for prev in reversed(previous_lines[-10:]):
        carrier, conf, matched = detect_carrier(prev)
        if carrier:
            return carrier, conf, f"context_line:{matched}"

    return None, 0.0, "not_found"


def create_offer_candidates(slide_lines_df: pd.DataFrame, slide_sections_df: pd.DataFrame):
    section_lookup = slide_sections_df.set_index("slide_id").to_dict("index")

    records = []
    sorted_df = slide_lines_df.sort_values(["page_number", "line_number_on_page"]).reset_index(drop=True)

    previous_lines_by_page = {}

    for idx, row in sorted_df.iterrows():
        page_number = row["page_number"]
        text = row["cleaned_line_text"]

        if page_number not in previous_lines_by_page:
            previous_lines_by_page[page_number] = []

        if not is_offer_like_line(text):
            previous_lines_by_page[page_number].append(text)
            continue

        section_info = section_lookup.get(row["slide_id"], {})
        section_name = section_info.get("predicted_section_name", "unknown")
        business_segment = section_info.get("business_segment", "unknown")

        previous_context = previous_lines_by_page[page_number]
        context_before = previous_context[-1] if previous_context else ""

        context_after = ""
        if idx < len(sorted_df) - 1:
            context_after = sorted_df.loc[idx + 1, "cleaned_line_text"]

        carrier, carrier_conf, carrier_reason = infer_carrier_from_context(text, previous_context)

        offer_subcategory, offer_conf, matched_offer_kw = detect_offer_subcategory(text)
        offer_subcategory = offer_subcategory or "unknown"
        offer_category = infer_offer_category(offer_subcategory)

        money_values = detect_money_values(text)
        device_models = extract_device_models(text)
        dates = detect_dates(text)

        offer_group_id = make_hash(f"{row['slide_id']}|{carrier}|{section_name}|{context_before}|{text}")
        candidate_id = make_hash(f"{offer_group_id}|{row['line_id']}")

        records.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "candidate_id": candidate_id,
            "offer_group_id": offer_group_id,
            "slide_id": row["slide_id"],
            "line_id": row["line_id"],
            "page_number": page_number,
            "section_name": section_name,
            "business_segment": business_segment,
            "carrier": carrier,
            "raw_offer_text": text,
            "context_before_text": context_before,
            "context_after_text": context_after,
            "suggested_offer_category": offer_category,
            "suggested_offer_subcategory": offer_subcategory,
            "detected_money_values_json": safe_json(money_values),
            "detected_device_models_json": safe_json(device_models),
            "detected_dates_json": safe_json(dates),
            "candidate_confidence": max(carrier_conf, offer_conf),
            "candidate_source": f"dictionary_regex|carrier_reason={carrier_reason}|offer_kw={matched_offer_kw}",
            "created_at": datetime.utcnow()
        })

        previous_lines_by_page[page_number].append(text)

    return pd.DataFrame(records)


silver_offer_candidates_df = create_offer_candidates(bronze_slide_lines_df, silver_slide_sections_df)

preview_df(silver_offer_candidates_df, "silver_offer_candidates_df", rows=30)

append_df_to_bq(silver_offer_candidates_df, TABLES["silver_offerCandidates"])

print("STEP 15 completed: Offer candidates loaded.")

In [ ]:
# ============================================================
# 16. Normalize offers
# ============================================================

print("\nSTEP 16: Normalizing offer candidates...")

def infer_offer_type(offer_subcategory: str, text: str, flags: dict):
    if offer_subcategory == "device_discount" and flags["aal_required_flag"]:
        return "aal_device_discount"
    if offer_subcategory == "device_discount" and flags["port_required_flag"]:
        return "port_device_discount"
    if offer_subcategory == "device_discount":
        return "device_discount"
    if offer_subcategory == "trade_in_offer":
        return "trade_in_discount"
    if offer_subcategory == "free_device_offer":
        return "free_device"
    if offer_subcategory == "monthly_bill_credit":
        return "monthly_bill_credit"
    if offer_subcategory == "byod_credit":
        return "byod_credit"
    if offer_subcategory == "eip_buyout":
        return "device_payoff_credit"
    if offer_subcategory == "plan_refresh":
        return "new_or_updated_plan"
    if offer_subcategory == "price_increase":
        return "monthly_plan_price_increase"
    if offer_subcategory == "activation_fee_waiver":
        return "activation_fee_waiver"
    if offer_subcategory == "expired_offer":
        return "expired_offer"
    if offer_subcategory == "no_change":
        return "no_change"
    return "unknown"


def infer_customer_action(flags: dict):
    actions = []
    if flags["port_required_flag"]:
        actions.append("port")
    if flags["aal_required_flag"]:
        actions.append("add_a_line")
    if flags["trade_required_flag"]:
        actions.append("trade_in")
    if flags["upgrade_required_flag"]:
        actions.append("upgrade")
    if flags["new_line_required_flag"]:
        actions.append("new_line")

    return ",".join(actions) if actions else None


def extract_plan_name(text: str):
    patterns = [
        r"Unlimited Ultimate",
        r"Unlimited Plus",
        r"Unlimited Welcome",
        r"Extra 2\.0",
        r"Premium 2\.0",
        r"Value 2\.0",
        r"Any UNL",
        r"Any Unlimited",
        r"most plans",
        r"SuperMobile",
        r"ProMobile",
        r"CoreMobile",
        r"Mobile Plus",
        r"Mobile Select"
    ]

    for pattern in patterns:
        match = re.search(pattern, text or "", flags=re.IGNORECASE)
        if match:
            return match.group(0)

    return None


def extract_condition_text(text: str):
    if not text:
        return None
    if " w/ " in text:
        return "Requires " + text.split(" w/ ", 1)[1]
    return None


def calculate_confidence(carrier, offer_category, offer_subcategory, product_family, offer_value, source_text):
    score = 0.0

    if carrier:
        score += 0.20
    if offer_category and offer_category != "unknown":
        score += 0.20
    if offer_subcategory and offer_subcategory != "unknown":
        score += 0.20
    if product_family:
        score += 0.15
    if offer_value is not None or "free" in source_text.lower() or "no changes" in source_text.lower():
        score += 0.15
    if len(source_text) > 10:
        score += 0.10

    return round(min(score, 1.0), 2)


def validate_offer(row_dict: dict):
    if FORCE_ALL_ROWS_TO_REVIEW:
        return "forced_review", True

    if row_dict["normalization_confidence"] < NORMALIZATION_CONFIDENCE_THRESHOLD:
        return "low_confidence", True

    if not row_dict["carrier"] and row_dict["offer_subcategory"] != "no_change":
        return "missing_carrier", True

    if row_dict["offer_category"] == "unknown":
        return "unknown_category", True

    if len(detect_money_values(row_dict["source_raw_text"])) > 2:
        return "multiple_money_values_review", True

    return "passed", False


def normalize_candidate(row: pd.Series):
    text = row["raw_offer_text"]
    lower = text.lower()

    manufacturer, _, _ = detect_manufacturer(text)
    product_family, _, _ = detect_product_family(text)

    device_models = json.loads(row["detected_device_models_json"] or "[]")
    primary_device_model = device_models[0] if device_models else None
    device_family_text = ", ".join(device_models) if device_models else None

    money_values = json.loads(row["detected_money_values_json"] or "[]")
    offer_value = normalize_money_value(money_values[0]) if money_values else None

    offer_value_unit = "usd" if offer_value is not None else None
    monthly_price = None

    if "/mo" in lower or "/month" in lower:
        monthly_price = offer_value
        offer_value_unit = "usd_per_month"

    dates = json.loads(row["detected_dates_json"] or "[]")
    effective_start_date, effective_end_date = parse_date_range(dates[0], year) if dates else (None, None)

    term_length_months = detect_term_months(text)

    flags = {
        "port_required_flag": any(x in lower for x in ["port", "port-in"]),
        "aal_required_flag": any(x in lower for x in ["aal", "add a line"]),
        "trade_required_flag": any(x in lower for x in ["trade", "trade-in", "trd", "fmv"]),
        "upgrade_required_flag": any(x in lower for x in ["upgrade", "upg"]),
        "new_line_required_flag": any(x in lower for x in ["new line", "new customer"]),
        "online_only_flag": "online only" in lower,
        "in_store_only_flag": "in-store" in lower or "in store" in lower,
        "is_free_flag": any(x in lower for x in ["free", "on us", "$0"]),
        "is_new_flag": any(x in lower for x in ["new", "launched", "introduced"]),
        "is_expired_flag": any(x in lower for x in ["expired", "removed", "retired", "ended"]),
    }

    offer_category = row["suggested_offer_category"] or "unknown"
    offer_subcategory = row["suggested_offer_subcategory"] or "unknown"
    offer_type = infer_offer_type(offer_subcategory, text, flags)

    normalization_confidence = calculate_confidence(
        carrier=row["carrier"],
        offer_category=offer_category,
        offer_subcategory=offer_subcategory,
        product_family=product_family,
        offer_value=offer_value,
        source_text=text
    )

    base = {
        "deck_id": deck_id,
        "run_id": run_id,
        "pdf_hash": pdf_hash,
        "pdf_name": pdf_name,
        "deck_week": deck_week,
        "offer_id": make_hash(f"{row['candidate_id']}|{offer_category}|{offer_subcategory}|{offer_value}|{text}"),
        "offer_group_id": row["offer_group_id"],
        "candidate_id": row["candidate_id"],
        "slide_id": row["slide_id"],
        "line_id": row["line_id"],
        "page_number": row["page_number"],
        "section_name": row["section_name"],
        "business_segment": row["business_segment"],
        "carrier": row["carrier"],
        "customer_segment": "business" if row["business_segment"] == "business" else "consumer",
        "product_family": product_family,
        "manufacturer": manufacturer,
        "device_family_text": device_family_text,
        "primary_device_model": primary_device_model,
        "device_models_json": safe_json(device_models),
        "plan_name": extract_plan_name(text),
        "offer_category": offer_category,
        "offer_subcategory": offer_subcategory,
        "offer_type": offer_type,
        "offer_value": offer_value,
        "offer_value_unit": offer_value_unit,
        "monthly_price": monthly_price,
        "data_allowance": (detect_data_allowance(text)[0].upper().replace(" ", "") if detect_data_allowance(text) else None),
        "term_length_months": term_length_months,
        "customer_action": infer_customer_action(flags),
        "condition_text": extract_condition_text(text),
        "port_required_flag": flags["port_required_flag"],
        "aal_required_flag": flags["aal_required_flag"],
        "trade_required_flag": flags["trade_required_flag"],
        "upgrade_required_flag": flags["upgrade_required_flag"],
        "new_line_required_flag": flags["new_line_required_flag"],
        "online_only_flag": flags["online_only_flag"],
        "in_store_only_flag": flags["in_store_only_flag"],
        "is_free_flag": flags["is_free_flag"],
        "is_new_flag": flags["is_new_flag"],
        "is_expired_flag": flags["is_expired_flag"],
        "effective_start_date": effective_start_date,
        "effective_end_date": effective_end_date,
        "source_raw_text": text,
        "category_confidence": row["candidate_confidence"],
        "normalization_confidence": normalization_confidence,
        "validation_status": None,
        "review_required_flag": None,
        "review_status": None,
        "status": None,
        "created_at": datetime.utcnow()
    }

    validation_status, review_required = validate_offer(base)

    base["validation_status"] = validation_status
    base["review_required_flag"] = review_required

    if review_required:
        base["review_status"] = "pending_review"
        base["status"] = "needs_review"
    else:
        base["review_status"] = "auto_approved" if AUTO_APPROVE_HIGH_CONFIDENCE else "pending_review"
        base["status"] = "ready_for_gold" if AUTO_APPROVE_HIGH_CONFIDENCE else "needs_review"

    return base


normalized_records = []

for _, row in silver_offer_candidates_df.iterrows():
    normalized_records.append(normalize_candidate(row))

silver_normalized_offers_df = pd.DataFrame(normalized_records)

preview_df(silver_normalized_offers_df, "silver_normalized_offers_df", rows=30)

append_df_to_bq(silver_normalized_offers_df, TABLES["silver_normalizedOffers"])

print("STEP 16 completed: Normalized offers loaded.")

In [ ]:
# ============================================================
# 17. Create offer-device bridge
# ============================================================

print("\nSTEP 17: Creating offer-device bridge...")

def create_offer_device_bridge(normalized_df: pd.DataFrame):
    records = []

    for _, row in normalized_df.iterrows():
        devices = json.loads(row["device_models_json"] or "[]")

        if not devices:
            continue

        for idx, device in enumerate(devices, start=1):
            manufacturer, _, _ = detect_manufacturer(device)

            records.append({
                "deck_id": deck_id,
                "run_id": run_id,
                "pdf_hash": pdf_hash,
                "pdf_name": pdf_name,
                "deck_week": deck_week,
                "offer_device_id": make_hash(f"{row['offer_id']}|{device}|{idx}"),
                "offer_id": row["offer_id"],
                "offer_group_id": row["offer_group_id"],
                "page_number": row["page_number"],
                "manufacturer": manufacturer or row["manufacturer"],
                "device_model": device,
                "device_rank": idx,
                "source_raw_text": row["source_raw_text"],
                "created_at": datetime.utcnow()
            })

    return pd.DataFrame(records)


silver_offer_device_bridge_df = create_offer_device_bridge(silver_normalized_offers_df)

preview_df(silver_offer_device_bridge_df, "silver_offer_device_bridge_df", rows=30)

append_df_to_bq(silver_offer_device_bridge_df, TABLES["silver_offerDeviceBridge"])

print("STEP 17 completed: Offer-device bridge loaded.")

In [ ]:
# ============================================================
# 18. Create review queue and auto approvals
# ============================================================

print("\nSTEP 18: Creating review queue and auto approvals...")

def create_review_queue(normalized_df: pd.DataFrame):
    records = []

    review_df = normalized_df[
        (normalized_df["review_required_flag"] == True) |
        (normalized_df["review_status"] == "pending_review")
    ].copy()

    print(f"Rows requiring human review: {len(review_df)}")

    for _, row in review_df.iterrows():
        review_id = make_hash(f"{row['offer_id']}|review")

        records.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "review_id": review_id,
            "offer_id": row["offer_id"],
            "offer_group_id": row["offer_group_id"],
            "candidate_id": row["candidate_id"],
            "page_number": row["page_number"],
            "section_name": row["section_name"],
            "business_segment": row["business_segment"],
            "carrier": row["carrier"],
            "source_raw_text": row["source_raw_text"],
            "suggested_output_json": safe_json(row.to_dict()),
            "confidence_score": row["normalization_confidence"],
            "validation_status": row["validation_status"],
            "review_reason": row["validation_status"],
            "review_status": "pending",
            "created_at": datetime.utcnow()
        })

    return pd.DataFrame(records)


def create_auto_approval_decisions(normalized_df: pd.DataFrame):
    records = []

    auto_df = normalized_df[
        (normalized_df["review_status"] == "auto_approved") &
        (normalized_df["status"] == "ready_for_gold")
    ].copy()

    print(f"Rows auto-approved by system: {len(auto_df)}")

    for _, row in auto_df.iterrows():
        review_id = make_hash(f"{row['offer_id']}|auto_review")
        decision_id = make_hash(f"{review_id}|auto_approved")

        records.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "decision_id": decision_id,
            "review_id": review_id,
            "offer_id": row["offer_id"],
            "offer_group_id": row["offer_group_id"],
            "decision_type": "auto_approved",
            "review_status": "approved",
            "reviewer_name": "system",
            "review_notes": "Auto-approved because validation passed and confidence threshold was met.",
            "final_output_json": safe_json(row.to_dict()),
            "created_at": datetime.utcnow()
        })

    return pd.DataFrame(records)


review_extraction_review_df = create_review_queue(silver_normalized_offers_df)
review_auto_decisions_df = create_auto_approval_decisions(silver_normalized_offers_df)

preview_df(review_extraction_review_df, "review_extraction_review_df", rows=30)
preview_df(review_auto_decisions_df, "review_auto_decisions_df", rows=30)

append_df_to_bq(review_extraction_review_df, TABLES["review_extractionReview"])
append_df_to_bq(review_auto_decisions_df, TABLES["review_reviewDecisions"])

print("STEP 18 completed: Review queue and auto approvals loaded.")

In [ ]:
# ============================================================
# 19. QA summary
# ============================================================

print("\nSTEP 19: Running QA summary...")

print("\nSlide sections:")
display(
    silver_slide_sections_df
    .groupby(["predicted_section_name", "business_segment", "classification_source"])
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

print("\nNormalized offers by section/status:")
display(
    silver_normalized_offers_df
    .groupby(["section_name", "business_segment", "validation_status", "review_status", "status"])
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

print("\nOffer categories:")
display(
    silver_normalized_offers_df
    .groupby(["offer_category", "offer_subcategory", "offer_type"])
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

print("\nRows needing review:")
display(
    silver_normalized_offers_df[
        silver_normalized_offers_df["status"] == "needs_review"
    ][[
        "page_number",
        "section_name",
        "business_segment",
        "carrier",
        "offer_category",
        "offer_subcategory",
        "validation_status",
        "normalization_confidence",
        "source_raw_text"
    ]].head(50)
)

print("\nMulti-device grouped offers:")
display(
    silver_normalized_offers_df[
        silver_normalized_offers_df["device_models_json"].str.contains(",", na=False)
    ][[
        "page_number",
        "carrier",
        "device_family_text",
        "device_models_json",
        "offer_type",
        "offer_value",
        "source_raw_text"
    ]].head(50)
)

print("\nFinal counts:")
print(f"PDF pages processed: {total_pages}")
print(f"Bronze slide raw rows: {len(bronze_slide_raw_df)}")
print(f"Bronze slide line rows: {len(bronze_slide_lines_df)}")
print(f"Detected entity rows: {len(bronze_detected_entities_df)}")
print(f"Silver slide section rows: {len(silver_slide_sections_df)}")
print(f"Silver offer candidate rows: {len(silver_offer_candidates_df)}")
print(f"Silver normalized offer rows: {len(silver_normalized_offers_df)}")
print(f"Device bridge rows: {len(silver_offer_device_bridge_df)}")
print(f"Review queue rows: {len(review_extraction_review_df)}")
print(f"Auto-approved decision rows: {len(review_auto_decisions_df)}")

print("\nSTEP 19 completed: QA summary generated.")
print("=" * 100)
print("Notebook 1 completed successfully.")
print("=" * 100)